# ***CIFAR10 Project***

In [3]:
# %pip install torchvision

import torch
import torch.nn as nn
import torch.optim as optim

import torchvision
from torchvision.datasets import CIFAR10

In [5]:
# Datasets & DataLoaders
from torch.utils.data import DataLoader
import torchvision.transforms as transforms

transform = transforms.Compose([
    transforms.ToTensor(), # convert values in tensors and scale to [0, 1]
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)) # normalize to [-1, 1]
])

trainset = CIFAR10(root="./data", train=True, download=True, transform=transform)
testset = CIFAR10(root="./data", train=False, download=True, transform=transform)

In [6]:
trainloader = DataLoader(trainset, batch_size=64, shuffle=True)
testloader = DataLoader(testset, batch_size=64)

# ***Building the CNN***

In [7]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()

        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2), # kernel size = 2, stride = 2

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2) ,

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )

        self.fc_layers = nn.Sequential(
            nn.Linear(4*4*128, 256),
            nn.ReLU(),

            nn.Linear(256, 10)
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1) # flattening
        x = self.fc_layers(x)

        return x

In [8]:
model = CNN()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

# ***Training the CNN***

In [14]:
epochs = 10
val_losses = []
best_val_loss = float("inf")

for epoch in range(epochs):

    # -------- TRAIN --------
    model.train()
    running_train_loss = 0.0

    for images, labels in trainloader:
        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_train_loss += loss.item()

    epoch_train_loss = running_train_loss / len(trainloader)


    # -------- VALIDATION --------
    model.eval()
    running_val_loss = 0.0

    with torch.no_grad():
        for images, labels in testloader:
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_val_loss += loss.item()

    epoch_val_loss = running_val_loss / len(testloader)
    val_losses.append(epoch_val_loss)

    print(f"Epoch {epoch+1}/{epochs} | train loss: {epoch_train_loss:.4f} | val loss: {epoch_val_loss:.4f}")

    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        torch.save(model.state_dict(), "best_model.pt")

Epoch 1/10 | train loss: 0.1371 | val loss: 1.1957
Epoch 2/10 | train loss: 0.1207 | val loss: 1.2227
Epoch 3/10 | train loss: 0.0960 | val loss: 1.3210
Epoch 4/10 | train loss: 0.1014 | val loss: 1.4128
Epoch 5/10 | train loss: 0.0846 | val loss: 1.5019
Epoch 6/10 | train loss: 0.0893 | val loss: 1.4921
Epoch 7/10 | train loss: 0.0758 | val loss: 1.7338
Epoch 8/10 | train loss: 0.0769 | val loss: 1.6670
Epoch 9/10 | train loss: 0.0614 | val loss: 1.8683
Epoch 10/10 | train loss: 0.0721 | val loss: 1.7231


# ***Evaluate Our CNN***

In [15]:

correct_labels = 0
total_labels = 0

model.eval()

with torch.no_grad():
    for images, labels in testloader:
        outputs = model.forward(images)
        _, predicted  = torch.max(outputs, 1)

        correct_labels += (predicted == labels).sum().item()
        total_labels += labels.size(0)

print(f"accuracy = {correct_labels / total_labels * 100}")

accuracy = 74.65
